In [1]:
import sys

import pandas as pd

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts

from dice4el.scenario.scenario_handler import ScenarioHandler
from dice4el.scenario.scenario_model import ScenarioLSTM, train_ScenarioLSTM, validate_ScenarioLSTM

### --- Load Dataset ---

In [2]:
set_seed(seed=42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
excel_file = pd.ExcelFile("../../../data/bpic19.xlsx", engine="openpyxl")

df = pd.concat(
    [
        pd.read_excel(
            excel_file,
            keep_default_na=False,
            dtype={
                "case:concept:name": "string",
                "concept:name": "string",
                "case:Spend area text": "string",
                "case:Document Type": "string",
                "case:Sub spend area text": "string",
                "case:Purch. Doc. Category name": "string",
                "case:Item Type": "string",
                "case:Item Category": "string",
                "case:Spend classification text": "string",
                "case:Source": "string",
                "case:GR-Based Inv. Verif.": "string",
                "case:Goods Receipt": "string",
                "Cumulative net worth (EUR)": "float32",
                "time_delta": "float32",
            }
        )
        for sheet in excel_file.sheet_names
    ],
    ignore_index=True,
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,Cumulative net worth (EUR),case:Document Type,case:GR-Based Inv. Verif.,case:Goods Receipt,case:Item Category,case:Item Type,case:Purch. Doc. Category name,case:Source,case:Spend area text,case:Spend classification text,case:Sub spend area text,concept:name,time_delta
0,2000000000_00001,2018-01-02 12:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Created,0.0
1,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Complete,3600.0
2,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Awaiting Approval,0.0
3,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Document Completed,0.0
4,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: In Transfer to Execution Syst.,0.0
5,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Ordered,0.0
6,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Change was Transmitted,0.0
7,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Create Purchase Order Item,0.0
8,2000000000_00001,2018-01-02 22:59:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Vendor creates invoice,32760.0
9,2000000000_00001,2018-03-06 06:44:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Record Goods Receipt,5384700.0


In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['Cumulative net worth (EUR)', 'case:Document Type', 'case:GR-Based Inv. Verif.', 'case:Goods Receipt', 'case:Item Category', 'case:Item Type', 'case:Purch. Doc. Category name', 'case:Source', 'case:Spend area text', 'case:Spend classification text', 'case:Sub spend area text', 'concept:name', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
case:Document Type             categorical    case     yes    ['EC Purchase order', 'Framework order', 'Standard PO'] N/A        data_derived        
case:GR-Based Inv. Verif.      categorical    case     yes    ['False', 'True

In [7]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

### --- Scenario Model ---

In [8]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [9]:
scenario_df = scenario_handler.generate_scenario_df(
    df=df,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
    n_scenarios_per_length=3
)

In [10]:
scenario_df.head()

,case:concept:name,time_index,fake,Cumulative net worth (EUR),case:Document Type,case:GR-Based Inv. Verif.,case:Goods Receipt,case:Item Category,case:Item Type,case:Purch. Doc. Category name,case:Source,case:Spend area text,case:Spend classification text,case:Sub spend area text,concept:name,time_delta
0,f_330928,0,True,3565.147988,EC Purchase order,True,True,2-way match,Service,Purchase order,sourceSystemID_0000,Spend Area Unidentified,PR,Public Relations,Vendor creates invoice,5.000463e+05
1,f_330928,1,True,369.266356,EC Purchase order,True,True,2-way match,Service,Purchase order,sourceSystemID_0000,Spend Area Unidentified,PR,Public Relations,SRM: Created,7.803842e+05
2,f_330928,2,True,155.044284,EC Purchase order,True,True,2-way match,Service,Purchase order,sourceSystemID_0000,Spend Area Unidentified,PR,Public Relations,Change Delivery Indicator,8.487934e+05
3,f_330928,3,True,81.443310,EC Purchase order,True,True,2-way match,Service,Purchase order,sourceSystemID_0000,Spend Area Unidentified,PR,Public Relations,SRM: Ordered,1.082051e+05
4,f_330928,4,True,898.455638,EC Purchase order,True,True,2-way match,Service,Purchase order,sourceSystemID_0000,Spend Area Unidentified,PR,Public Relations,Change Price,1.061939e+06


In [11]:
case_ids = scenario_df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = scenario_df[scenario_df["case:concept:name"].isin(train_cases)].copy()
val_df   = scenario_df[scenario_df["case:concept:name"].isin(val_cases)].copy()

In [12]:
train_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=train_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [13]:
val_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=val_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [14]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [15]:
criterion = torch.nn.BCEWithLogitsLoss()

In [16]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic19-scenario_model_output.txt")

Epoch 020/100 | Train Loss: 0.0001 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.0000 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.0000 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.0000 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.0000 | LR: 1.00e-06
Time taken for scenario model (training): 33624.266304 seconds
Time taken for scenario model (validation): 50.477469 seconds
Val loss: {'loss': 0.000136411105393268, 'accuracy': 0.999990477324128, 'f1_macro': 0.9999872986829924, 'f1_weighted': 0.9999904773014381}


In [17]:
embedding_metadata = scenario_handler.get_scenario_embedding_metadata()

scenario_model = ScenarioLSTM(
    categorical_info=embedding_metadata["categorical_info"],
    n_continuous=embedding_metadata["n_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ScenarioLSTM(
    model=scenario_model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

scenario_model.save()

In [18]:
scenario_model = ScenarioLSTM.load()

In [19]:
val_loss = validate_ScenarioLSTM(
    model=scenario_model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [20]:
sys.stdout = original_stdout
log_file.close()